# DỰ BÁO GIÁ CỔ PHIẾU VCB — ỨNG DỤNG N-BEATS
**Chuyên ngành:** Hệ thống Thông tin Quản lý (MIT) — UEH  
**Học viên:** Phan Minh Trí | **Năm:** 2026

## Cấu trúc Notebook
| Phase | Nội dung | Cells |
|-------|----------|-------|
| 1 | Setup: Cài đặt thư viện, Import & Cấu hình | 1–2 |
| 2 | Data: Thu thập, EDA, Chia tập | 3–5 |
| 3 | Models: Naive → ARIMA → LSTM → N-BEATS → N-BEATSx | 6–10 |
| 4 | Analysis: So sánh, Phần dư, Diebold-Mariano, Sensitivity | 11–14 |
| 5 | Output: Xuất kết quả, DSS Prototype | 15–16 |



---
## Phase 1 — Setup
### Cell 1: Cài đặt thư viện

In [ ]:
# ============================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN
# ============================================================
!pip install -q -U vnstock
!pip install -q neuralforecast==1.7.4
!pip install -q statsforecast==1.7.4
!pip install -q pmdarima
!pip install -q yfinance
!pip install -q scikit-learn matplotlib seaborn statsmodels
print('✅ Xong! Hãy RESTART RUNTIME rồi chạy từ Cell 2.')


### Cell 2: Import & Cấu hình toàn cục

In [ ]:
# ============================================================
# CELL 2: IMPORT & CẤU HÌNH TOÀN CỤC
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS, NBEATSx

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import yfinance as yf
from pathlib import Path

# --- Matplotlib ---
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False
})
COLORS = ['#2563EB', '#DC2626', '#16A34A', '#D97706', '#7C3AED']

# ============================================================
# THAM SỐ TOÀN CỤC
# ============================================================
TICKER      = 'VCB'
START_DATE  = '2015-01-01'
END_DATE    = '2025-12-31'
HORIZON     = 5
LOOKBACK    = 30
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
SEED        = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'✅ Cấu hình: {TICKER} | {START_DATE} → {END_DATE}')
print(f'   Horizon={HORIZON}d | Lookback={LOOKBACK}d | Split 70/15/15')

---
## Phase 2 — Dữ liệu


### Cell 3: Thu thập dữ liệu (VCB + USD/VND + VNINDEX)

In [ ]:
# ==============================
# CELL 3: Lấy dữ liệu
# - VCB từ yfinance: giá đóng cửa điều chỉnh (Adj Close)
# - USD/VND từ yfinance (VND=X)
# - VNINDEX từ vnstock (KBS)
# ==============================

import pandas as pd
import yfinance as yf
from vnstock import Quote

# ──────────────────────────────────────────────────────────────
# 1. Hàm lấy dữ liệu từ yfinance
# ──────────────────────────────────────────────────────────────
def get_yf_history(ticker, start_date="2015-01-01", end_date="2026-01-01"):
    df = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        interval="1d",
        auto_adjust=False,   # Giữ cột Adj Close riêng biệt
        progress=False
    )
    if df is None or df.empty:
        raise ValueError(f"Không lấy được dữ liệu cho {ticker} từ yfinance")

    df = df.reset_index()

    # Xử lý MultiIndex columns (yfinance >= 0.2.x)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] if col[0] else col[1] for col in df.columns]

    df["date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None)

    keep_cols = ["date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].copy()

    rename_map = {
        "Open": "open", "High": "high", "Low": "low",
        "Close": "close", "Adj Close": "adj_close", "Volume": "volume"
    }
    df = df.rename(columns=rename_map)
    df = df.sort_values("date").reset_index(drop=True)
    return df


# ──────────────────────────────────────────────────────────────
# 2. Hàm lấy dữ liệu VNINDEX từ vnstock (KBS)
# ──────────────────────────────────────────────────────────────
def get_kbs_history(symbol, start_date="2015-01-01", end_date="2025-12-31"):
    quote = Quote(symbol=symbol, source="KBS")
    df = quote.history(start=start_date, end=end_date, interval="1D")
    if df is None or df.empty:
        raise ValueError(f"Không lấy được dữ liệu cho {symbol} từ KBS")

    df = df.copy()
    df.columns = [str(c).lower() for c in df.columns]

    if "time" in df.columns:
        df["date"] = pd.to_datetime(df["time"]).dt.tz_localize(None)
    elif "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    else:
        raise ValueError(f"{symbol}: Không tìm thấy cột thời gian")

    keep_cols = ["date", "open", "high", "low", "close", "volume"]
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].sort_values("date").reset_index(drop=True)
    return df


# ──────────────────────────────────────────────────────────────
# 3. Lấy giá VCB  (giá đóng cửa điều chỉnh)
# ──────────────────────────────────────────────────────────────
print("📥 [1/3] Tải VCB (adj close) từ yfinance...")
df_vcb = get_yf_history(ticker="VCB.VN", start_date="2015-01-01", end_date="2026-01-01")

if "adj_close" not in df_vcb.columns:
    raise ValueError("Không tìm thấy cột Adj Close trong dữ liệu VCB.")

print(f"   ✅ VCB: {len(df_vcb)} phiên | {df_vcb['date'].min().date()} → {df_vcb['date'].max().date()}")
print(f"   Adj Close  – mean={df_vcb['adj_close'].mean():,.0f}  "      f"min={df_vcb['adj_close'].min():,.0f}  max={df_vcb['adj_close'].max():,.0f}")
print(f"   Close gốc  – mean={df_vcb['close'].mean():,.0f}  "      f"min={df_vcb['close'].min():,.0f}  max={df_vcb['close'].max():,.0f}")

# ──────────────────────────────────────────────────────────────
# 4. Lấy VNINDEX từ vnstock (KBS)
# ──────────────────────────────────────────────────────────────
print("\n📥 [2/3] Tải VNINDEX từ vnstock (KBS)...")
df_vnindex = get_kbs_history(symbol="VNINDEX", start_date="2015-01-01", end_date="2025-12-31")
print(f"   ✅ VNINDEX: {len(df_vnindex)} phiên | "      f"{df_vnindex['date'].min().date()} → {df_vnindex['date'].max().date()}")
print(f"   Close – mean={df_vnindex['close'].mean():,.1f}  "      f"min={df_vnindex['close'].min():,.1f}  max={df_vnindex['close'].max():,.1f}")

# ──────────────────────────────────────────────────────────────
# 5. Lấy tỷ giá USD/VND từ yfinance
# ──────────────────────────────────────────────────────────────
print("\n📥 [3/3] Tải tỷ giá USD/VND từ yfinance...")
usd_vnd = get_yf_history(ticker="VND=X", start_date="2015-01-01", end_date="2026-01-01")
print(f"   ✅ USD/VND: {len(usd_vnd)} ngày | "      f"{usd_vnd['date'].min().date()} → {usd_vnd['date'].max().date()}")
print(f"   Close – mean={usd_vnd['close'].mean():,.0f}  "      f"min={usd_vnd['close'].min():,.0f}  max={usd_vnd['close'].max():,.0f}")

# ──────────────────────────────────────────────────────────────
# 6. Export 3 file CSV riêng
# ──────────────────────────────────────────────────────────────
df_vcb.to_csv("VCB_2015_2025_adjusted.csv",   index=False, encoding="utf-8-sig")
df_vnindex.to_csv("VNINDEX_2015_2025.csv",     index=False, encoding="utf-8-sig")
usd_vnd.to_csv("USDVND_2015_2025.csv",         index=False, encoding="utf-8-sig")
print("\n✅ Đã lưu 3 file CSV riêng biệt.")

# ──────────────────────────────────────────────────────────────
# 7. Chuẩn hoá date và merge
# ──────────────────────────────────────────────────────────────
df_vcb["date"]     = pd.to_datetime(df_vcb["date"]).dt.date
df_vnindex["date"] = pd.to_datetime(df_vnindex["date"]).dt.date
usd_vnd["date"]    = pd.to_datetime(usd_vnd["date"]).dt.date

# Biến mục tiêu: giá đóng cửa điều chỉnh, đặt tên vcb_close
vcb_close    = df_vcb[["date", "adj_close"]].rename(columns={"adj_close": "vcb_close"})
vnindex_close = df_vnindex[["date", "close"]].rename(columns={"close": "vnindex_close"})
usdvnd_close  = usd_vnd[["date", "close"]].rename(columns={"close": "usdvnd_close"})

df_merged = (
    vcb_close
    .merge(vnindex_close, on="date", how="inner")
    .merge(usdvnd_close,  on="date", how="inner")
    .sort_values("date")
    .reset_index(drop=True)
)

# ──────────────────────────────────────────────────────────────
# 8. Kiểm tra dữ liệu sau merge
# ──────────────────────────────────────────────────────────────
print("\n=== Kiểm tra dữ liệu sau merge ===")
print(f"Shape: {df_merged.shape}")
print(f"Khoảng thời gian: {df_merged['date'].min()} → {df_merged['date'].max()}")
print("Missing values:")
print(df_merged.isna().sum())
print("Duplicated date:", df_merged.duplicated(subset=["date"]).sum())

# Kiểm tra nhanh giá trị bất thường
for col, lo, hi in [
    ("vcb_close",    1_000,  200_000),
    ("vnindex_close",  200,    3_000),
    ("usdvnd_close", 15_000, 30_000),
]:
    n_out = ((df_merged[col] < lo) | (df_merged[col] > hi)).sum()
    flag  = "✅" if n_out == 0 else f"⚠️  {n_out} giá trị ngoài khoảng [{lo:,}–{hi:,}]"
    print(f"  {col}: {flag}")

# ──────────────────────────────────────────────────────────────
# 9. Export file gộp
# ──────────────────────────────────────────────────────────────
df_merged.to_csv(
    "Merged_VCB_ADJCLOSE_VNINDEX_USDVND_2015_2025.csv",
    index=False, encoding="utf-8-sig"
)
print("\n✅ Đã lưu file gộp: Merged_VCB_ADJCLOSE_VNINDEX_USDVND_2015_2025.csv")
print(df_merged.head(3).to_string(index=False))
print("...")
print(df_merged.tail(3).to_string(index=False))


### Cell 4: Phân tích khám phá dữ liệu (EDA)

In [ ]:
# ============================================================
# CELL 4: EDA
# Bổ sung: (1) Thống kê mô tả  (2) Phân tích mùa vụ
# ============================================================
import scipy.stats as stats

df = df_merged.copy()
df['date'] = pd.to_datetime(df['date'])

# ──────────────────────────────────────────────────────────────
# 4.1  Thống kê mô tả
# ──────────────────────────────────────────────────────────────
print("=" * 62)
print("  THỐNG KÊ MÔ TẢ")
print("=" * 62)

desc_rows = []
for col, label in [
    ("vcb_close",    "VCB Adj Close (VNĐ)"),
    ("vnindex_close","VNINDEX (điểm)"),
    ("usdvnd_close", "USD/VND"),
]:
    s = df[col]
    desc_rows.append({
        "Biến"       : label,
        "Obs"        : len(s),
        "Mean"       : s.mean(),
        "Std"        : s.std(),
        "Min"        : s.min(),
        "25%"        : s.quantile(0.25),
        "Median"     : s.median(),
        "75%"        : s.quantile(0.75),
        "Max"        : s.max(),
        "Skewness"   : stats.skew(s.dropna()),
        "Kurtosis"   : stats.kurtosis(s.dropna()),
    })

df_desc = pd.DataFrame(desc_rows).set_index("Biến")
print(df_desc.round(2).to_string())

# ──────────────────────────────────────────────────────────────
# 4.2  Biểu đồ tổng quan 3 biến + dual-axis
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 15))

for ax, col, title, color in [
    (axes[0], 'vcb_close',    'Giá điều chỉnh VCB – Adj Close (2015–2025)',  COLORS[0]),
    (axes[1], 'usdvnd_close', 'Tỷ giá USD/VND (2015–2025)',                  COLORS[1]),
    (axes[2], 'vnindex_close','Chỉ số VNINDEX (2015–2025)',                  COLORS[4]),
]:
    ax.plot(df['date'], df[col], color=color, linewidth=1)
    ax.fill_between(df['date'], df[col], alpha=0.1, color=color)
    ax.set_title(title, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax  = axes[3]
ax2 = ax.twinx()
ax.plot(df['date'],  df['vcb_close'],    color=COLORS[0], lw=1.2, label='VCB (trái)')
ax2.plot(df['date'], df['vnindex_close'],color=COLORS[4], lw=1.2, ls='--', label='VNINDEX (phải)')
ax.set_title('Tương quan: VCB vs VNINDEX', fontweight='bold')
ax.set_ylabel('Giá VCB Adj Close (VNĐ)', color=COLORS[0])
ax2.set_ylabel('VNINDEX', color=COLORS[4])
lines = ax.get_lines() + ax2.get_lines()
ax.legend(lines, [l.get_label() for l in lines], loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Tổng quan dữ liệu VCB (Adj Close), USD/VND, VNINDEX',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_overview.png', dpi=130, bbox_inches='tight')
plt.show()

# ──────────────────────────────────────────────────────────────
# 4.3  Ma trận tương quan + Heatmap
# ──────────────────────────────────────────────────────────────
corr = df[['vcb_close','usdvnd_close','vnindex_close']].corr()
print('\n📊 Ma trận tương quan (Pearson):')
print(corr.round(4))
print(f'\n→ r(VCB, USD/VND) = {corr.loc["vcb_close","usdvnd_close"]:.4f}')
print(f'→ r(VCB, VNINDEX) = {corr.loc["vcb_close","vnindex_close"]:.4f}')
print(f'→ r(USD/VND, VNINDEX) = {corr.loc["usdvnd_close","vnindex_close"]:.4f}')

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt='.4f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, square=True)
ax.set_title('Heatmap tương quan', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_corr_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()

# ──────────────────────────────────────────────────────────────
# 4.4  Phân tích thành phần mùa vụ (Seasonal Decomposition)
# ──────────────────────────────────────────────────────────────
from statsmodels.tsa.seasonal import seasonal_decompose

# Cần index là DatetimeIndex liên tục (ffill khoảng trống cuối tuần)
vcb_series = df.set_index('date')['vcb_close']
vcb_series.index = pd.to_datetime(vcb_series.index)
vcb_resampled = vcb_series.resample('B').last().ffill()  # Business day

decomp = seasonal_decompose(
    vcb_resampled,
    model='additive',
    period=252,              # ~252 phiên giao dịch / năm
    extrapolate_trend='freq'
)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
components = [
    (decomp.observed, 'Observed (Quan sát)',   COLORS[0]),
    (decomp.trend,    'Trend (Xu hướng)',       COLORS[1]),
    (decomp.seasonal, 'Seasonal (Mùa vụ)',      COLORS[2]),
    (decomp.resid,    'Residual (Phần dư)',      COLORS[3]),
]
for ax, (comp, title, color) in zip(axes, components):
    ax.plot(comp.index, comp.values, color=color, linewidth=0.9)
    ax.set_title(title, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    if title.startswith('Residual'):
        ax.axhline(0, color='gray', linewidth=0.8, ls='--')

plt.suptitle('Phân tích thành phần chuỗi thời gian VCB (period=252 phiên/năm)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_seasonal_decomp.png', dpi=130, bbox_inches='tight')
plt.show()
print("✅ EDA hoàn tất")


# ──────────────────────────────────────────────────────────────
# 4.5  Kiểm định tính dừng – ADF Test
# ──────────────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

def adf_test(series, name):
    res = adfuller(series.dropna(), autolag='AIC')
    status = '✅ Dừng (p < 0.05)' if res[1] < 0.05 else '❌ Không dừng'
    print(f"  [{name}]  stat={res[0]:7.3f}  p={res[1]:.4f}  "          f"Crit_5%={res[4]['5%']:.3f}  → {status}")
    return res[1]

print('\n=== Kiểm định ADF – Chuỗi gốc ===')
for col, name in [
    ('vcb_close',    'VCB Adj Close'),
    ('usdvnd_close', 'USD/VND'),
    ('vnindex_close','VNINDEX'),
]:
    adf_test(df[col], name)

print('\n=== Kiểm định ADF – Sai phân bậc 1 (Δ1) ===')
for col, name in [
    ('vcb_close',    'VCB Δ1'),
    ('usdvnd_close', 'USD/VND Δ1'),
    ('vnindex_close','VNINDEX Δ1'),
]:
    adf_test(df[col].diff().dropna(), name)

# ──────────────────────────────────────────────────────────────
# 4.6  ACF / PACF
# ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf( df['vcb_close'].dropna(), lags=40, ax=axes[0],
          color=COLORS[0], title='ACF – Giá VCB Adj Close')
plot_pacf(df['vcb_close'].dropna(), lags=40, ax=axes[1],
          color=COLORS[1], title='PACF – Giá VCB Adj Close')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_acf_pacf.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Kiểm định tính dừng hoàn tất')


# ============================================================
# 4.6: PHÂN TÍCH TƯƠNG QUAN CHÉO CÓ ĐỘ TRỄ
# ============================================================
from scipy.stats import pearsonr

vcb_arr = df_merged['vcb_close'].values
vni_arr = df_merged['vnindex_close'].values
fx_arr  = df_merged['usdvnd_close'].values
MAX_LAG = 30

print('=== Tương quan chéo có độ trễ: VNINDEX → VCB ===')
print(f'{"Lag":>6} | {"r":>8} | {"p-value":>10} | Ý nghĩa')
print('-' * 45)
vni_ccf = []
for lag in range(MAX_LAG + 1):
    if lag == 0:
        r, p = pearsonr(vni_arr, vcb_arr)
    else:
        r, p = pearsonr(vni_arr[:-lag], vcb_arr[lag:])
    vni_ccf.append(r)
    if lag <= 10 or lag == 20 or lag == 30:
        sig = '✅' if p < 0.05 else '  '
        trend = '→ đồng thời' if lag == 0 else f'→ lag {lag}d'
        print(f'{lag:>6} | {r:>8.4f} | {p:>10.6f} | {sig} {trend}')

print()
print('=== Tương quan chéo có độ trễ: USD/VND → VCB ===')
fx_ccf = []
for lag in range(MAX_LAG + 1):
    if lag == 0:
        r, p = pearsonr(fx_arr, vcb_arr)
    else:
        r, p = pearsonr(fx_arr[:-lag], vcb_arr[lag:])
    fx_ccf.append(r)
    if lag <= 10 or lag == 20 or lag == 30:
        sig = '✅' if p < 0.05 else '  '
        print(f'{lag:>6} | {r:>8.4f} | {p:>10.6f} | {sig}')

# ── Biểu đồ Cross-correlation ──
import numpy as np
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

lags = list(range(MAX_LAG + 1))
axes[0].bar(lags, vni_ccf, color=COLORS[4], alpha=0.85, width=0.7)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].axhline(0.05,  color='gray', lw=0.7, ls='--', alpha=0.7)
axes[0].axhline(-0.05, color='gray', lw=0.7, ls='--', alpha=0.7)
axes[0].set_title('Cross-correlation: VNINDEX(t−lag) vs VCB(t)', fontweight='bold')
axes[0].set_xlabel('Lag (ngày giao dịch)')
axes[0].set_ylabel('Hệ số tương quan Pearson r')
axes[0].set_xticks([0,5,10,15,20,25,30])
axes[0].annotate(f'r(lag=0)={vni_ccf[0]:.3f}', xy=(0, vni_ccf[0]),
                 xytext=(3, vni_ccf[0]+0.02), fontsize=9)

axes[1].bar(lags, fx_ccf, color=COLORS[1], alpha=0.85, width=0.7)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Cross-correlation: USD/VND(t−lag) vs VCB(t)', fontweight='bold')
axes[1].set_xlabel('Lag (ngày giao dịch)')
axes[1].set_xticks([0,5,10,15,20,25,30])

plt.suptitle('Phân tích tương quan chéo có độ trễ — Biến ngoại sinh vs Giá VCB',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cross_correlation_lag.png', dpi=130, bbox_inches='tight')
plt.show()

# ── Nhận xét ──
vni_max_lag = int(np.argmax(np.abs(vni_ccf[1:]))) + 1
fx_max_lag  = int(np.argmax(np.abs(fx_ccf[1:]))) + 1
print(f'\n=== Nhận xét ===')
print(f'VNINDEX: r giảm dần theo lag (r_max ngoài lag=0 tại lag={vni_max_lag}d, r={vni_ccf[vni_max_lag]:.4f})')
print(f'USD/VND: r giảm dần theo lag (r_max ngoài lag=0 tại lag={fx_max_lag}d, r={fx_ccf[fx_max_lag]:.4f})')
print()
if vni_ccf[0] > vni_ccf[1] and vni_ccf[0] > vni_ccf[5]:
    print('→ VNINDEX có tương quan CAO NHẤT tại lag=0 (đồng thời)')
    print('  Không phải leading indicator → giải thích N-BEATSx không cải thiện N-BEATS')
    print('  Biến ngoại sinh phản ánh thông tin đồng thời, không phải tín hiệu trước VCB')
print('✅ Cross-correlation analysis hoàn tất')


### Cell 5: Chia tập & Chuẩn hóa

In [ ]:
# ============================================================
# CELL 5: CHIA TẬP & CHUẨN HÓA
# ============================================================

n       = len(df)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)
n_test  = n - n_train - n_val

df_train = df.iloc[:n_train].copy()
df_val   = df.iloc[n_train:n_train+n_val].copy()
df_test  = df.iloc[n_train+n_val:].copy()

print(f'Train: n={n_train} | {df_train["date"].iloc[0]} → {df_train["date"].iloc[-1]}')
print(f'Val  : n={n_val}   | {df_val["date"].iloc[0]} → {df_val["date"].iloc[-1]}')
print(f'Test : n={n_test}  | {df_test["date"].iloc[0]} → {df_test["date"].iloc[-1]}')

# --- Chuẩn hóa (fit chỉ trên train) ---
sc_vcb  = MinMaxScaler()
sc_fx   = MinMaxScaler()
sc_vni  = MinMaxScaler()

tr_vcb = sc_vcb.fit_transform(df_train[['vcb_close']]).flatten()
va_vcb = sc_vcb.transform(df_val[['vcb_close']]).flatten()
te_vcb = sc_vcb.transform(df_test[['vcb_close']]).flatten()

tr_fx  = sc_fx.fit_transform(df_train[['usdvnd_close']]).flatten()
va_fx  = sc_fx.transform(df_val[['usdvnd_close']]).flatten()
te_fx  = sc_fx.transform(df_test[['usdvnd_close']]).flatten()

tr_vni = sc_vni.fit_transform(df_train[['vnindex_close']]).flatten()
va_vni = sc_vni.transform(df_val[['vnindex_close']]).flatten()
te_vni = sc_vni.transform(df_test[['vnindex_close']]).flatten()

all_vcb_sc = np.concatenate([tr_vcb, va_vcb, te_vcb])

# --- Hàm tạo sequence ---
def make_seq_1var(arr, lookback=LOOKBACK, horizon=HORIZON):
    X, y = [], []
    for i in range(lookback, len(arr) - horizon + 1):
        X.append(arr[i-lookback:i])
        y.append(arr[i:i+horizon])
    return np.array(X), np.array(y)

# --- Hàm đánh giá ---
RESULTS = {}   # {model_name: {RMSE, MAE, MAPE, R2}}
PREDS   = {}   # {model_name: (actual_vnd_array, pred_vnd_array, dates_array)}
             # — cấu trúc nhất quán cho TẤT CẢ 5 mô hình

def evaluate(y_true, y_pred, name):
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100
    r2   = r2_score(y_true, y_pred)
    print(f'\n📊 [{name}]  RMSE={rmse:,.2f}  MAE={mae:,.2f}  MAPE={mape:.4f}%  R²={r2:.6f}')
    return {'Model': name, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

# Vẽ split
fig, ax = plt.subplots(figsize=(14,4))
for subset, label, c in [
    (df_train, f'Train (n={n_train})', COLORS[0]),
    (df_val,   f'Val   (n={n_val})',   COLORS[2]),
    (df_test,  f'Test  (n={n_test})',  COLORS[1])
]:
    ax.plot(subset['date'], subset['vcb_close'], color=c, label=label, lw=1.3)
ax.axvline(df_val['date'].iloc[0],  color='gray', ls='--', alpha=0.7)
ax.axvline(df_test['date'].iloc[0], color='gray', ls='--', alpha=0.7)
ax.set_title('Phân chia tập dữ liệu VCB (70/15/15)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'split.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Chuẩn bị hoàn tất')

---
## Phase 3 — Các mô hình dự báo


### Cell 6: Model 1 — Naive

In [ ]:
# ============================================================
# CELL 6: NAIVE MODEL (Lower Bound)
# Mục đích: Thiết lập ngưỡng tối thiểu (lower bound) để
#   đánh giá liệu các mô hình phức tạp có thực sự cải thiện.
#
# Naive: ŷ_{t+1} = ... = ŷ_{t+H} = y_t  (giá hôm nay)
# Dùng giá GỐC (VNĐ) – không qua scaled, nhất quán với các mô hình khác.
# ============================================================
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

vcb_actual = df_test['vcb_close'].values   # giá thực tế VNĐ
dates_test  = df_test['date'].values

naive_preds   = []
naive_actuals = []
naive_dates   = []

for i in range(0, len(vcb_actual) - HORIZON, HORIZON):
    last_val     = vcb_actual[i]
    pred_block   = np.full(HORIZON, last_val)
    actual_block = vcb_actual[i + 1 : i + 1 + HORIZON]
    date_block   = dates_test[i + 1 : i + 1 + HORIZON]
    if len(actual_block) == HORIZON:
        naive_preds.extend(pred_block)
        naive_actuals.extend(actual_block)
        naive_dates.extend(date_block)

naive_preds   = np.array(naive_preds)
naive_actuals = np.array(naive_actuals)

RESULTS['Naive'] = evaluate(naive_actuals, naive_preds, 'Naive')
PREDS['Naive']   = (naive_actuals, naive_preds, np.array(naive_dates))

# Sanity check
print(f'\nSanity check: Naive RMSE = {RESULTS["Naive"]["RMSE"]:,.0f} VNĐ')
print('→ Các mô hình phức tạp phải có RMSE < Naive để có giá trị')

# Biểu đồ
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(naive_dates, naive_actuals, color=COLORS[0], lw=1.5, label='Thực tế')
ax.plot(naive_dates, naive_preds,   color='gray',    lw=1.5, ls='--', label='Naive (ŷ = giá hôm nay)')
ax.set_title('Naive Model — Dự báo giá VCB (Tập Test)', fontweight='bold')
ax.set_ylabel('Giá (VNĐ)'); ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pred_naive.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Naive model hoàn tất')


### Cell 7: Model 2 — ARIMA (+ Ljung-Box)

In [ ]:
# ============================================================
# CELL 6: ARIMA
# ============================================================
import time
t0 = time.time()

train_val_vcb = np.concatenate([df_train['vcb_close'].values, df_val['vcb_close'].values])
test_vcb      = df_test['vcb_close'].values

print('🔍 Auto ARIMA...')
auto = pm.auto_arima(
    train_val_vcb,
    start_p=0, start_q=0, max_p=5, max_q=5,
    d=1, seasonal=False, information_criterion='aic',
    stepwise=True, trace=True,
    error_action='ignore', suppress_warnings=True
)
p, d, q = auto.order
print(f'✅ ARIMA({p},{d},{q}) | AIC={auto.aic():.2f}')

history = list(train_val_vcb)
preds_ar = []
n_win = len(test_vcb) // HORIZON
rem   = len(test_vcb) %  HORIZON

for i in range(n_win):
    fit = ARIMA(history, order=(p,d,q)).fit()
    preds_ar.extend(fit.forecast(steps=HORIZON))
    history.extend(test_vcb[i*HORIZON:(i+1)*HORIZON])
if rem:
    fit = ARIMA(history, order=(p,d,q)).fit()
    preds_ar.extend(fit.forecast(steps=rem))

preds_ar = np.array(preds_ar[:len(test_vcb)])
RESULTS['ARIMA'] = evaluate(test_vcb, preds_ar, 'ARIMA')
PREDS['ARIMA']   = (test_vcb, preds_ar, df_test['date'].values)

print(f'⏱️  {(time.time()-t0)/60:.1f} phút')

fig, ax = plt.subplots(figsize=(14,5))
ax.plot(df_test['date'], test_vcb, color=COLORS[0], lw=1.5, label='Thực tế')
ax.plot(df_test['date'], preds_ar, color=COLORS[1], lw=1.5, ls='--', label=f'ARIMA({p},{d},{q})')
ax.set_title(f'ARIMA({p},{d},{q}) – Dự báo giá VCB (Tập Test)', fontweight='bold')
ax.set_ylabel('Giá (VNĐ)'); ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pred_arima.png', dpi=130, bbox_inches='tight')
plt.show()

# ──────────────────────────────────────────────────────────────
# 6.2  Kiểm định Ljung-Box – Phần dư ARIMA
# Mục đích: xác nhận phần dư là white noise (model fit tốt)
# H0: phần dư không có tự tương quan (residuals ~ white noise)
# Kết quả mong muốn: p-value > 0.05 → chấp nhận H0
# ──────────────────────────────────────────────────────────────
from statsmodels.stats.diagnostic import acorr_ljungbox

# Fit lại ARIMA trên toàn bộ train+val để lấy phần dư
arima_final = ARIMA(train_val_vcb, order=(p, d, q)).fit()
residuals_arima = arima_final.resid

# Ljung-Box test tại các lag: 5, 10, 15, 20
lb_result = acorr_ljungbox(residuals_arima, lags=[5, 10, 15, 20], return_df=True)

print('\n=== Kiểm định Ljung-Box – Phần dư ARIMA({},{},{}) ==='.format(p,d,q))
print(f'  H0: Phần dư là white noise (không có tự tương quan)')
print(f'  Kết quả mong muốn: p-value > 0.05')
print()
print(f'  {"Lag":>6} | {"LB Statistic":>14} | {"p-value":>10} | Kết luận')
print('  ' + '-' * 55)
for lag, row in lb_result.iterrows():
    stat   = row['lb_stat']
    pval   = row['lb_pvalue']
    concl  = '✅ White noise' if pval > 0.05 else '❌ Còn tự tương quan'
    print(f'  {lag:>6} | {stat:>14.4f} | {pval:>10.4f} | {concl}')

# Biểu đồ phần dư ARIMA
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(residuals_arima, color=COLORS[1], linewidth=0.8, alpha=0.8)
axes[0].axhline(0, color='gray', linewidth=0.8, ls='--')
axes[0].set_title(f'Phần dư ARIMA({p},{d},{q})', fontweight='bold')
axes[0].set_xlabel('Thời gian (phiên)')
axes[0].set_ylabel('Phần dư')

axes[1].hist(residuals_arima, bins=40, color=COLORS[1], edgecolor='white',
             alpha=0.85, density=True)
import scipy.stats as stats_mod
mu, std = residuals_arima.mean(), residuals_arima.std()
x_range = np.linspace(mu - 4*std, mu + 4*std, 200)
axes[1].plot(x_range, stats_mod.norm.pdf(x_range, mu, std),
             color='black', linewidth=1.5, label='Normal PDF')
axes[1].set_title('Phân phối phần dư', fontweight='bold')
axes[1].set_xlabel('Phần dư')
axes[1].set_ylabel('Mật độ')
axes[1].legend()

from statsmodels.graphics.tsaplots import plot_acf as _plot_acf
_plot_acf(residuals_arima, lags=20, ax=axes[2],
          color=COLORS[0], title='ACF phần dư ARIMA')

plt.suptitle(f'Chẩn đoán phần dư – ARIMA({p},{d},{q})',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'arima_residual_diagnostic.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ Kiểm định Ljung-Box hoàn tất')


### Cell 8: Model 3 — LSTM

In [ ]:
import os, random
import numpy as np

os.environ['PYTHONHASHSEED']         = str(SEED)
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)

# ── BƯỚC 1: Import các thư viện còn lại ────────────────────
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print(f"✅ Random seed đã cố định: SEED={SEED}")
print(f"   TF version: {tf.__version__}")

# ── BƯỚC 2: Cấu hình mô hình ───────────────────────────────
CPU_CONFIG = {
    'lstm_units1'       : 128,
    'lstm_units2'       : 64,
    'lstm_patience'     : 20,
    'lstm_epochs'       : 200,
    'lstm_batch'        : 32,
    'dropout_rate'      : 0.2,
    'reduce_lr_patience': 8,
}

t0 = time.time()

# ── BƯỚC 3: Tạo sequences ──────────────────────────────────
# Val sequences lấy prefix từ cuối train để có đủ lookback
X_tr_s, y_tr_s = make_seq_1var(tr_vcb)
val_prefix      = np.concatenate([tr_vcb[-LOOKBACK:], va_vcb])
X_va_s, y_va_s = make_seq_1var(val_prefix)

X_tr_s = X_tr_s.reshape(-1, LOOKBACK, 1)
X_va_s = X_va_s.reshape(-1, LOOKBACK, 1)
print(f"\nTrain seq: {X_tr_s.shape} | Val seq: {X_va_s.shape}")

# ── BƯỚC 4: Kiến trúc LSTM ─────────────────────────────────
# Dùng kernel_initializer với seed cố định để đảm bảo
# khởi tạo trọng số hoàn toàn tái lập
ki = tf.keras.initializers.GlorotUniform(seed=SEED)
ri = tf.keras.initializers.Orthogonal(seed=SEED)

model_lstm = Sequential([
    LSTM(
        CPU_CONFIG['lstm_units1'],
        return_sequences=True,
        input_shape=(LOOKBACK, 1),
        kernel_initializer=ki,
        recurrent_initializer=ri,
    ),
    Dropout(CPU_CONFIG['dropout_rate'], seed=SEED),
    LSTM(
        CPU_CONFIG['lstm_units2'],
        return_sequences=False,
        kernel_initializer=ki,
        recurrent_initializer=ri,
    ),
    Dropout(CPU_CONFIG['dropout_rate'], seed=SEED),
    Dense(16, activation='relu', kernel_initializer=ki),
    Dense(HORIZON, kernel_initializer=ki),
])

model_lstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae'],
)
model_lstm.summary()

# ── BƯỚC 5: Training ───────────────────────────────────────
cbs = [
    EarlyStopping(
        monitor='val_loss',
        patience=CPU_CONFIG['lstm_patience'],
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=CPU_CONFIG['reduce_lr_patience'],
        min_lr=1e-6,
        verbose=0,
    ),
]

print(f"\n🚀 Training LSTM (max {CPU_CONFIG['lstm_epochs']} epochs, "
      f"batch={CPU_CONFIG['lstm_batch']}, SEED={SEED})...")

hist_lstm = model_lstm.fit(
    X_tr_s, y_tr_s,
    validation_data=(X_va_s, y_va_s),
    epochs=CPU_CONFIG['lstm_epochs'],
    batch_size=CPU_CONFIG['lstm_batch'],
    callbacks=cbs,
    verbose=0,
  )

# ── BƯỚC 6: Learning curve & Gap ratio ────────────────────
final_train = hist_lstm.history['loss'][-1]
final_val   = hist_lstm.history['val_loss'][-1]
gap_ratio   = final_val / (final_train + 1e-10)
stopped_ep  = len(hist_lstm.history['loss'])

print(f"\n📊 Kết quả training:")
print(f"   Epochs chạy thực : {stopped_ep}")
print(f"   Train loss        : {final_train:.6f}")
print(f"   Val loss          : {final_val:.6f}")
print(f"   Gap ratio         : {gap_ratio:.2f}x  "
      f"({'✅ Tốt (<5x)' if gap_ratio < 5 else '⚠️  Overfitting' if gap_ratio < 20 else '❌ Overfitting nặng'})")

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist_lstm.history['loss'],     label='Train Loss', color=COLORS[0])
axes[0].plot(hist_lstm.history['val_loss'], label='Val Loss',   color=COLORS[1])
axes[0].set_title('LSTM – Learning Curve', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()

axes[1].bar(
    ['Train Loss', 'Val Loss'],
    [final_train, final_val],
    color=[COLORS[0], COLORS[1]],
    width=0.5, edgecolor='white',
)
axes[1].set_title(f'Loss cuối – Gap ratio = {gap_ratio:.2f}x', fontweight='bold')
for i, v in enumerate([final_train, final_val]):
    axes[1].text(i, v * 1.02, f'{v:.5f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'lstm_loss.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'⏱️  Training LSTM: {(time.time()-t0)/60:.1f} phút')

# ── BƯỚC 7: Dự báo trên tập Test (Rolling Direct) ─────────
# Dùng chuỗi đã chuẩn hóa toàn bộ; rolling theo bước H
full_scaled    = all_vcb_sc   # train+val+test đã chuẩn hóa
test_start_idx = n_train + n_val

lstm_preds_scaled = []
n_test_windows    = n_test // HORIZON
remainder_test    = n_test %  HORIZON

for i in range(n_test_windows):
    idx     = test_start_idx + i * HORIZON
    x_input = full_scaled[idx - LOOKBACK : idx].reshape(1, LOOKBACK, 1)
    pred    = model_lstm.predict(x_input, verbose=0)[0]
    lstm_preds_scaled.extend(pred)

if remainder_test > 0:
    idx     = test_start_idx + n_test_windows * HORIZON
    x_input = full_scaled[idx - LOOKBACK : idx].reshape(1, LOOKBACK, 1)
    pred    = model_lstm.predict(x_input, verbose=0)[0]
    lstm_preds_scaled.extend(pred[:remainder_test])

# Inverse transform về đơn vị VNĐ
lstm_preds_scaled = np.array(lstm_preds_scaled[:n_test]).reshape(-1, 1)
lstm_preds        = sc_vcb.inverse_transform(lstm_preds_scaled).flatten()
test_actual       = df_test['vcb_close'].values

# ── BƯỚC 8: Đánh giá ──────────────────────────────────────
RESULTS['LSTM'] = evaluate(test_actual, lstm_preds, 'LSTM')
PREDS['LSTM']   = (test_actual, lstm_preds, df_test['date'].values)

print(f"\n📈 LSTM Results:")
for k, v in RESULTS['LSTM'].items():
    # Check if the value is numeric before applying float formatting
    if isinstance(v, (int, float, np.number)):
        print(f"   {k}: {v:,.4f}")
    else:
        print(f"   {k}: {v}") # Print as string

# ── BƯỚC 9: Biểu đồ dự báo ────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_test['date'].values, test_actual,
        color=COLORS[0], lw=1.5, label='Thực tế')
ax.plot(df_test['date'].values, lstm_preds,
        color=COLORS[1], lw=1.5, ls='--', label='LSTM dự báo')
ax.set_title('LSTM – Dự báo giá VCB (Tập Test)', fontweight='bold')
ax.set_ylabel('Giá VCB (VNĐ)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'lstm_pred.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'\n✅ LSTM hoàn tất — SEED={SEED} — Kết quả hoàn toàn tái lập')

### Cell 9a: Phân tích độ nhạy — Tìm Lookback Tối Ưu

1.   Mục danh sách
2.   Mục danh sách


> **Mục đích:** Xác định `LOOKBACK` tối ưu bằng thực nghiệm **trước khi** chạy N-BEATS.
> Kết quả này biện minh cho việc chọn `LOOKBACK = 30` trong Cell 9 và 10.

In [ ]:
# ============================================================
# CELL 9: SENSITIVITY ANALYSIS – LOOKBACK WINDOW
# Mục đích: kiểm tra LOOKBACK=30 có phải tối ưu không
# Chạy N-BEATS với 4 lookback: 10, 20, 30, 60 ngày
# ============================================================
import time
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd # Import pandas here if not already available
import sys # Import sys to modify recursion limit

# Increase the recursion limit - default is 1000
sys.setrecursionlimit(3000)

# Define df_nf and nf_test_size, which are used by NeuralForecast
df_nf = pd.DataFrame({
    'unique_id': TICKER,
    'ds': pd.to_datetime(df['date']),
    'y' : df['vcb_close'].astype(float)
})

nf_test_size = n_test - (n_test % HORIZON)

LOOKBACK_LIST  = [10, 20, 30, 60]
sensitivity_res = []

print('=== Sensitivity Analysis: Lookback Window ===')
print(f'{'Lookback':>10} | {'RMSE (VNĐ)':>14} | {'MAPE (%)':>10} | {'R²':>8} | {'Thời gian':>10}')
print('-' * 65)

for lb in LOOKBACK_LIST:
    try:
        t0 = time.time()
        nbeats_lb = NBEATS(
            h=HORIZON, input_size=lb,
            max_steps=500,
            batch_size=32,
            learning_rate=0.001,
            val_check_steps=50,
            early_stop_patience_steps=10,
            scaler_type='standard',
            random_seed=SEED,
        )
        nf_lb = NeuralForecast(models=[nbeats_lb], freq='B')
        nf_lb.fit(df_nf, val_size=n_val)

        cv_lb = nf_lb.cross_validation(
            df=df_nf, val_size=n_val,
            test_size=nf_test_size,
            n_windows=None, step_size=HORIZON
        )
        pred_col = [c for c in cv_lb.columns if 'NBEATS' in c.upper() and c != 'y'][0]
        cv_lb = cv_lb.groupby('ds').last().reset_index().sort_values('ds')

        merged_lb = (
            df_test[['date','vcb_close']].rename(columns={'date':'ds'})
            .merge(cv_lb[['ds', pred_col]], on='ds', how='inner')
            .dropna()
        )
        act_lb  = merged_lb['vcb_close'].values
        pred_lb = merged_lb[pred_col].values

        rmse_lb = np.sqrt(mean_squared_error(act_lb, pred_lb))
        mape_lb = np.mean(np.abs((act_lb - pred_lb) / (np.abs(act_lb) + 1e-8))) * 100
        r2_lb   = r2_score(act_lb, pred_lb)
        elapsed = time.time() - t0

        marker = ' ← Đã chọn' if lb == LOOKBACK else ''
        print(f'{lb:>10} | {rmse_lb:>14,.2f} | {mape_lb:>10.4f} | {r2_lb:>8.4f} | {elapsed:>8.1f}s{marker}')
        sensitivity_res.append({'Lookback': lb, 'RMSE': rmse_lb, 'MAPE': mape_lb, 'R2': r2_lb})
    except RecursionError:
        print(f'{lb:>10} | {"Error":>14} | {"Error":>10} | {"Error":>8} | {"Error":>10} -- RecursionError occurred')
    except Exception as e:
        print(f'{lb:>10} | {"Error":>14} | {"Error":>10} | {"Error":>8} | {"Error":>10} -- An unexpected error occurred: {e}')

import pandas as pd
df_sens = pd.DataFrame(sensitivity_res)
# Ensure df_sens is not empty before proceeding
if not df_sens.empty:
    best_lb = df_sens.loc[df_sens.RMSE.idxmin(), 'Lookback']
    print(f'\nLookback tối ưu theo RMSE: {int(best_lb)} ngày')

    if int(best_lb) == LOOKBACK:
        print(f'✅ LOOKBACK={LOOKBACK} ngày đã chọn là tối ưu — được xác nhận bởi thực nghiệm')
    else:
        print(f'⚠ LOOKBACK tối ưu là {int(best_lb)} ngày (khác với {LOOKBACK} đã dùng)')
        # Check if LOOKBACK is in df_sens before accessing
        if LOOKBACK in df_sens['Lookback'].values:
            pct_diff = (df_sens.loc[df_sens.Lookback==LOOKBACK,'RMSE'].values[0] - df_sens.RMSE.min()) / df_sens.RMSE.min() * 100
            print(f'  Chênh lệch: {pct_diff:.2f}% — nếu nhỏ có thể bỏ qua về mặt thực tiễn')
        else:
            print(f'  LOOKBACK={LOOKBACK} was skipped due to an error.')


    # ── Biểu đồ sensitivity ──
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(df_sens['Lookback'], df_sens['RMSE'], 'o-', color=COLORS[0], lw=2, ms=8)
    axes[0].axvline(LOOKBACK, color='red', lw=1.5, ls='--', label=f'Đã chọn ({LOOKBACK}d)', alpha=0.8)
    axes[0].set_xlabel('Lookback (ngày)')
    axes[0].set_ylabel('RMSE (VNĐ)')
    axes[0].set_title('RMSE theo Lookback Window', fontweight='bold')
    axes[0].set_xticks(LOOKBACK_LIST)
    axes[0].legend()
    for _, row in df_sens.iterrows():
        axes[0].annotate(f"{row['RMSE']:,.0f}", (row['Lookback'], row['RMSE']),
                         textcoords='offset points', xytext=(0,8), ha='center', fontsize=9)

    axes[1].plot(df_sens['Lookback'], df_sens['MAPE'], 's-', color=COLORS[2], lw=2, ms=8)
    axes[1].axvline(LOOKBACK, color='red', lw=1.5, ls='--', alpha=0.8)
    axes[1].set_xlabel('Lookback (ngày)')
    axes[1].set_ylabel('MAPE (%)')
    axes[1].set_title('MAPE theo Lookback Window', fontweight='bold')
    axes[1].set_xticks(LOOKBACK_LIST)

    plt.suptitle('Sensitivity Analysis: Hiệu suất N-BEATS theo Lookback Window',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'sensitivity_lookback.png', dpi=130, bbox_inches='tight')
    plt.show()
else:
    print("\n⚠️ No results were collected for sensitivity analysis.")

print('✅ Sensitivity analysis hoàn tất')


### Cell 9: Model 4 — N-BEATS (Mô hình trọng tâm)

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
# ============================================================
# CELL 9: N-BEATS
# ============================================================
t0 = time.time()

df_nf = pd.DataFrame({
    'unique_id': TICKER,
    'ds': pd.to_datetime(df['date']),
    'y' : df['vcb_close'].astype(float)
})

nf_test_size = n_test - (n_test % HORIZON)

nbeats = NBEATS(
    h=HORIZON, input_size=LOOKBACK,
    max_steps=1000, batch_size=32,
    learning_rate=1e-3, val_check_steps=50,
    early_stop_patience_steps=10,
    scaler_type='standard', random_seed=SEED
)
nf1 = NeuralForecast(models=[nbeats], freq='B')
print('🚀 Training N-BEATS...')
nf1.fit(df_nf, val_size=n_val)

cv1 = nf1.cross_validation(
    df=df_nf, val_size=n_val, test_size=nf_test_size,
    n_windows=None, step_size=HORIZON
)
cv1 = cv1.groupby('ds').last().reset_index().sort_values('ds')

merged_nb = pd.merge(
    df_test[['date','vcb_close']].assign(date=lambda x: pd.to_datetime(x['date'])).rename(columns={'date':'ds'}),
    cv1[['ds','NBEATS']], on='ds', how='inner'
).dropna()

RESULTS['NBEATS'] = evaluate(merged_nb['vcb_close'].values, merged_nb['NBEATS'].values, 'N-BEATS')
PREDS['NBEATS']   = (merged_nb['vcb_close'].values, merged_nb['NBEATS'].values, merged_nb['ds'].values)

print(f'⏱️  {(time.time()-t0)/60:.1f} phút')

fig, ax = plt.subplots(figsize=(14,5))
ax.plot(merged_nb['ds'], merged_nb['vcb_close'], color=COLORS[0], lw=1.5, label='Thực tế')
ax.plot(merged_nb['ds'], merged_nb['NBEATS'],    color=COLORS[2], lw=1.5, ls='--', label='N-BEATS')
ax.set_title('N-BEATS – Dự báo giá VCB (Tập Test)', fontweight='bold')
ax.set_ylabel('Giá (VNĐ)'); ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pred_nbeats.png', dpi=130, bbox_inches='tight')
plt.show()

### Cell 10: Model 5 — N-BEATSx (Biến ngoại sinh USD/VND + VNINDEX)

In [ ]:
import time
import pandas as pd

# ============================================================
# CELL 10: N-BEATSx với 2 biến ngoại sinh
#   hist_exog_list = ['usdvnd', 'vnindex']
# ============================================================
t0 = time.time()

# --- 10.1 DataFrame với cả 2 biến ngoại sinh ---
df_nfx = pd.DataFrame({
    'unique_id': TICKER,
    'ds'       : pd.to_datetime(df['date']), # Chuyển đổi sang datetime64[ns]
    'y'        : df['vcb_close'].astype(float),
    'usdvnd'   : df['usdvnd_close'].astype(float),    # Biến ngoại sinh 1
    'vnindex'  : df['vnindex_close'].astype(float)     # Biến ngoại sinh 2 (MỚI)
})

print('📋 NBEATSx dataset:')
print(df_nfx.tail(3))
print(f'\nBiến ngoại sinh: usdvnd + vnindex')
print(f'Loại: hist_exog (lịch sử, không cần giá trị tương lai)')

# --- 10.2 Khởi tạo N-BEATSx ---
nbeatsx = NBEATSx(
    h=HORIZON,
    input_size=LOOKBACK,
    hist_exog_list=['usdvnd', 'vnindex'],   # ← 2 biến ngoại sinh
    max_steps=1000,
    batch_size=32,
    learning_rate=1e-3,
    val_check_steps=50,
    early_stop_patience_steps=10,
    scaler_type='standard',
    random_seed=SEED
)

nf2 = NeuralForecast(models=[nbeatsx], freq='B')
print('\n🚀 Training N-BEATSx (USD/VND + VNINDEX)...')
nf2.fit(df_nfx, val_size=n_val)
print('✅ N-BEATSx training hoàn tất')

# --- 9.3 Dự báo N-BEATSx ---
cv2 = nf2.cross_validation(
    df=df_nfx, val_size=n_val, test_size=nf_test_size,
    n_windows=None, step_size=HORIZON
)
cv2 = cv2.groupby('ds').last().reset_index().sort_values('ds')

# Tìm tên cột dự báo (có thể là 'NBEATSx' hoặc 'NBEATS')
pred_col = [c for c in cv2.columns
            if 'NBEATS' in c.upper() and c != 'y'][0]
print(f'Cột dự báo N-BEATSx: {pred_col}')

merged_nbx = pd.merge(
    df_test[['date','vcb_close']].assign(date=lambda x: pd.to_datetime(x['date'])).rename(columns={'date':'ds'}),
    cv2[['ds', pred_col]], on='ds', how='inner'
).dropna()

RESULTS['NBEATSx'] = evaluate(
    merged_nbx['vcb_close'].values,
    merged_nbx[pred_col].values,
    'N-BEATSx (USD/VND + VNINDEX)'
)
PREDS['NBEATSx'] = (
    merged_nbx['vcb_close'].values,
    merged_nbx[pred_col].values,
    merged_nbx['ds'].values
)

print(f'⏱️  {(time.time()-t0)/60:.1f} phút')

fig, ax = plt.subplots(figsize=(14,5))
ax.plot(merged_nbx['ds'], merged_nbx['vcb_close'],  color=COLORS[0], lw=1.5, label='Thực tế')
ax.plot(merged_nbx['ds'], merged_nbx[pred_col],     color=COLORS[3], lw=1.5, ls='--',
        label='N-BEATSx (USD/VND + VNINDEX)')
ax.set_title('N-BEATSx (2 biến ngoại sinh: USD/VND + VNINDEX) – Dự báo VCB', fontweight='bold')
ax.set_ylabel('Giá (VNĐ)'); ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pred_nbeatsx_v2.png', dpi=130, bbox_inches='tight')
plt.show()

---
## Phase 4 — Phân tích kết quả


### Cell 11: So sánh tổng hợp 5 mô hình

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

MODEL_ORDER = ['Naive', 'ARIMA', 'LSTM', 'NBEATS', 'NBEATSx']
MODEL_LABEL = {
    'Naive'  : 'Naive (lower bound)',
    'ARIMA'  : 'ARIMA(0,1,0)',
    'LSTM'   : 'LSTM',
    'NBEATS' : 'N-BEATS ★',
    'NBEATSx': 'N-BEATSx (USD/VND + VNINDEX)',
}
BAR_COLORS  = ['gray', COLORS[3], COLORS[0], COLORS[2], COLORS[4]]

# ── Bảng kết quả ──
rows = []
for k in MODEL_ORDER:
    # Use the full set of evaluation results directly from RESULTS for all models
    r = RESULTS[k].copy()

    rows.append({
        'Mô hình': MODEL_LABEL[k],
        'RMSE (VNĐ)': round(r['RMSE'], 2),
        'MAE (VNĐ)' : round(r['MAE'],  2),
        'MAPE (%)'  : round(r['MAPE'],  4),
        'R²'        : round(r['R2'],    4),
    })

df_res = pd.DataFrame(rows).set_index('Mô hình')
print('=== Kết quả so sánh 5 mô hình trên tập Test ===')
print(df_res.to_string())

# ── Tính % cải thiện so với Naive ──
naive_rmse = RESULTS['Naive']['RMSE'] # Use RESULTS for Naive
print('\n=== Cải thiện so với Naive baseline ===')
for k in MODEL_ORDER[1:]:
    # Use the RMSE from RESULTS for all models
    current_rmse = RESULTS[k]['RMSE']
    imp = (naive_rmse - current_rmse) / naive_rmse * 100
    arrow = '↓ tốt hơn' if imp > 0 else '↑ kém hơn'
    print(f'  {MODEL_LABEL[k]:35} {imp:+.1f}% {arrow}')

# ── Biểu đồ 1: Bar chart RMSE ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

labels  = [MODEL_LABEL[k] for k in MODEL_ORDER]
rmses   = []
for k in MODEL_ORDER:
    rmses.append(RESULTS[k]['RMSE']) # Use RESULTS for all models
r2s     = []
for k in MODEL_ORDER:
    r2s.append(RESULTS[k]['R2']) # Use RESULTS for all models

axes[0].bar(labels, rmses, color=BAR_COLORS, edgecolor='white', width=0.6)
axes[0].axhline(naive_rmse, color='gray', lw=1, ls=':', alpha=0.7,
                label=f'Naive ({naive_rmse:,.0f}')
for i, v in enumerate(rmses):
    axes[0].text(i, v + 15, f'{v:,.0f}', ha='center', fontsize=8.5, fontweight='bold')
axes[0].set_title('RMSE (VNĐ) — thấp hơn = tốt hơn', fontweight='bold')
axes[0].set_ylabel('RMSE (VNĐ)')
axes[0].tick_params(axis='x', rotation=25)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[0].legend(fontsize=9)

axes[1].bar(labels, r2s, color=BAR_COLORS, edgecolor='white', width=0.6)
for i, v in enumerate(r2s):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=8.5, fontweight='bold')
axes[1].set_title('R² — cao hơn = tốt hơn', fontweight='bold')
axes[1].set_ylabel('R²')
axes[1].tick_params(axis='x', rotation=25)

plt.suptitle('So sánh hiệu suất 5 mô hình — Tập Test (10/05/2024–31/12/2025)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'compare_metrics.png', dpi=130, bbox_inches='tight')
plt.show()

# ── Biểu đồ 2: Đường dự báo từng mô hình ──
fig, axes = plt.subplots(5, 1, figsize=(14, 22), sharex=False)

for ax, key in zip(axes, MODEL_ORDER):
    # Use PREDS for the actual/pred arrays for consistency in plots
    actual, pred, dates = PREDS[key]
    ax.plot(dates, actual, color=COLORS[0], lw=1.5, label='Thực tế', alpha=0.9)
    c = 'gray' if key == 'Naive' else (
        COLORS[3] if key == 'ARIMA' else
        COLORS[0] if key == 'LSTM'  else
        COLORS[2] if key == 'NBEATS' else COLORS[4])

    # Use RESULTS for RMSE in the label for all models
    rmse_for_label = RESULTS[key]["RMSE"]

    ax.plot(dates, pred,   color=c, lw=1.5, ls='--',
            label=f'{MODEL_LABEL[key]}  RMSE={rmse_for_label:,.0f}')
    ax.set_title(MODEL_LABEL[key], fontweight='bold')
    ax.set_ylabel('Giá (VNĐ)')
    ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('Dự báo giá VCB — So sánh 5 mô hình (Tập Test)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'compare_all_5.png', dpi=130, bbox_inches='tight')
plt.show()
print('✅ So sánh tổng hợp hoàn tất')

### Cell 12: Phân tích phần dư (5 mô hình)

In [ ]:
# ============================================================
# CELL 12: PHÂN TÍCH PHẦN DƯ — 5 MÔ HÌNH
# residual = actual − predicted
# ============================================================
import numpy as np
import scipy.stats as stats_mod

# Ensure MODEL_ORDER, MODEL_LABEL, and BAR_COLORS are defined
# (These are typically defined in Cell 11 for consistency across analysis cells)
MODEL_ORDER = ['Naive', 'ARIMA', 'LSTM', 'NBEATS', 'NBEATSx']
MODEL_LABEL = {
    'Naive'  : 'Naive baseline',
    'ARIMA'  : 'ARIMA(0,1,0)',
    'LSTM'   : 'LSTM',
    'NBEATS' : 'N-BEATS ★',
    'NBEATSx': 'N-BEATSx (USD/VND + VNINDEX)',
}
BAR_COLORS  = ['gray', COLORS[3], COLORS[0], COLORS[2], COLORS[4]] # COLORS is assumed to be defined globally

fig, axes = plt.subplots(5, 3, figsize=(16, 22))

resid_stats = []

for row_idx, key in enumerate(MODEL_ORDER):
    actual, pred, dates = PREDS[key]
    resid = actual - pred
    mu    = resid.mean()
    std   = resid.std()
    skew  = stats_mod.skew(resid)
    kurt  = stats_mod.kurtosis(resid)
    _, pval_norm = stats_mod.normaltest(resid)

    resid_stats.append({
        'Mô hình'       : MODEL_LABEL[key],
        'Mean (VNĐ)'    : round(mu,   1),
        'Std (VNĐ)'     : round(std,  1),
        'Skewness'      : round(skew, 4),
        'Kurtosis'      : round(kurt, 4),
        'Normality p'   : round(pval_norm, 4),
    })

    c = 'gray' if key == 'Naive' else BAR_COLORS[MODEL_ORDER.index(key)]

    # (1) Residual theo thời gian
    ax0 = axes[row_idx, 0]
    ax0.plot(dates, resid, color=c, lw=0.9, alpha=0.85)
    ax0.axhline(0,    color='black', lw=0.8, ls='--')
    ax0.axhline(+2*std, color='gray', lw=0.6, ls=':')
    ax0.axhline(-2*std, color='gray', lw=0.6, ls=':')
    ax0.set_title(f'{MODEL_LABEL[key]} — Residual', fontsize=10, fontweight='bold')
    ax0.set_ylabel('Residual (VNĐ)')
    ax0.tick_params(axis='x', rotation=20, labelsize=8)

    # (2) Histogram + Normal PDF
    ax1 = axes[row_idx, 1]
    ax1.hist(resid, bins=35, color=c, edgecolor='white', alpha=0.8, density=True)
    x_r = np.linspace(mu - 4*std, mu + 4*std, 300)
    ax1.plot(x_r, stats_mod.norm.pdf(x_r, mu, std),
             color='black', lw=1.5, label='Normal PDF')
    ax1.set_title(f'Mean={mu:,.0f}  Std={std:,.0f}', fontsize=10, fontweight='bold')
    ax1.set_xlabel('Residual (VNĐ)')
    ax1.legend(fontsize=8)

    # (3) Q-Q Plot
    ax2 = axes[row_idx, 2]
    (osm, osr), (slope, intercept, _) = stats_mod.probplot(resid, dist='norm')
    ax2.scatter(osm, osr, color=c, s=8, alpha=0.6)
    x_line = np.array([osm[0], osm[-1]])
    ax2.plot(x_line, slope * x_line + intercept, color='black', lw=1.5)
    ax2.set_title(f'Q-Q Plot  (normality p={pval_norm:.4f})', fontsize=10, fontweight='bold')
    ax2.set_xlabel('Theoretical quantiles')
    ax2.set_ylabel('Sample quantiles')

plt.suptitle('Phân tích phần dư — 5 mô hình', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'residual_all_5.png', dpi=130, bbox_inches='tight')
plt.show()

# ── Bảng thống kê phần dư ──
df_resid = pd.DataFrame(resid_stats).set_index('Mô hình')
print('=== Thống kê phần dư — 5 mô hình ===')
print(df_resid.to_string())
print()
print('⚠ Kurtosis > 3 (fat tails) là đặc trưng dữ liệu tài chính — không phải dấu hiệu mô hình kém.')
print('  Tiêu chí đánh giá chính: RMSE, MAE, MAPE.')
print('✅ Phân tích phần dư hoàn tất')

### Cell 13: Kiểm định Diebold-Mariano

In [ ]:
# ============================================================
# CELL 13: KIỂM ĐỊNH DIEBOLD-MARIANO (DM Test)
# Diebold & Mariano (1995) + Harvey et al. (1997) small-sample
# H0: hai mô hình có độ chính xác tương đương
# Bác bỏ H0 khi p-value < 0.05
# ============================================================
import numpy as np
from scipy import stats

def diebold_mariano(e1, e2, h=1):
    d     = e1**2 - e2**2
    n     = len(d)
    d_bar = d.mean()
    gamma_0 = np.var(d, ddof=1)
    gamma_h_sum = sum(
        np.cov(d[k:], d[:-k])[0, 1] for k in range(1, h) if n - k > 1
    )
    lrv = gamma_0 + 2 * gamma_h_sum
    # Harvey et al. (1997) small-sample correction
    correction = (n + 1 - 2*h + h*(h-1)/n) / n
    lrv_c = max(lrv * correction, 1e-10)
    dm_stat = d_bar / np.sqrt(lrv_c / n)
    p_value = 2 * (1 - stats.t.cdf(abs(dm_stat), df=n - 1))
    return dm_stat, p_value

# ── Trích xuất sai số từ PREDS dict (nhất quán cho tất cả mô hình) ──
errors = {}

for key in MODEL_ORDER:
    actual, pred, dates = PREDS[key]
    errors[key] = np.asarray(actual, dtype=float) - np.asarray(pred, dtype=float)

print("=== DM Test alignment check ===")
for key in MODEL_ORDER:
    print(f"{key:8} | n_errors = {len(errors[key])}")

print('=== Kiểm định Diebold-Mariano ===')
print('H0: hai mô hình có độ chính xác tương đương')
print()
print(f'{"Cặp so sánh (M1 vs M2)":38} | {"DM Stat":>9} | {"p-value":>9} | Kết luận')
print('-' * 90)

dm_pairs = [
    ('NBEATS',  'Naive',  'N-BEATS vs Naive'),
    ('NBEATS',  'ARIMA',  'NBEATS vs ARIMA'),
    ('NBEATS',  'LSTM',   'N-BEATS vs LSTM'),
    ('NBEATS',  'NBEATSx','N-BEATS vs N-BEATSx'),
    ('NBEATSx', 'ARIMA',  'N-BEATSx vs ARIMA'),
    ('ARIMA',   'LSTM',   'ARIMA vs LSTM'),
    ('ARIMA',   'Naive',  'ARIMA vs Naive'),
]

dm_results = []
for m1, m2, label in dm_pairs:
    n = min(len(errors[m1]), len(errors[m2]))
    stat, pval = diebold_mariano(errors[m1][:n], errors[m2][:n], h=HORIZON)
    reject  = pval < 0.05
    better  = 'M1 tốt hơn ✅' if stat < 0 else 'M2 tốt hơn ✅'
    concl   = f'Bác bỏ H0 — {better}' if reject else 'Không bác bỏ H0 ❌'
    print(f'{label:38} | {stat:>9.3f} | {pval:>9.4f} | {concl}')
    dm_results.append({'pair': label, 'stat': stat, 'pval': pval, 'reject': reject})

# ── Biểu đồ p-values ──
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 4))
pair_labels = [r['pair'] for r in dm_results]
pvals       = [r['pval'] for r in dm_results]
bar_cols    = [COLORS[2] if p < 0.05 else '#CCCCCC' for p in pvals]

bars = ax.barh(pair_labels, pvals, color=bar_cols, alpha=0.85, height=0.55)
ax.axvline(0.05, color='red', lw=1.5, ls='--', label='α = 0.05 (ngưỡng bác bỏ H0)')
ax.set_xlabel('p-value')
ax.set_title('Kiểm định Diebold-Mariano (xanh = có ý nghĩa thống kê)', fontweight='bold')
ax.legend()
for bar, pval in zip(bars, pvals):
    ax.text(min(pval + 0.005, 0.8), bar.get_y() + bar.get_height()/2,
            f'{pval:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'diebold_mariano.png', dpi=130, bbox_inches='tight')
plt.show()

n_sig = sum(r['reject'] for r in dm_results)
print(f'\nSố cặp có ý nghĩa thống kê (p<0.05): {n_sig}/{len(dm_results)}')
print('✅ Kiểm định Diebold-Mariano hoàn tất')
